# 9.18 — Prompting (Zero/Few-Shot, Chain-of-Thought)

Prompting is the practical interface between a language model and the behavior we want: the same frozen model can answer differently when we change the instructions, demonstrations, scratch space, or output constraints placed in its context. In this lesson, we will build tiny pure-Python and NumPy substitutes for prompt templates, label logits, few-shot biases, chain-of-thought traces, and self-consistency votes so the mechanics are visible without any API calls or neural-network libraries.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build prompting one idea at a time. Run each cell in order and read the printed intermediate values — every piece of logic is small enough to inspect, and every probability comes from toy logits rather than a hidden model call. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, probabilities, and small simulations.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
from collections import Counter  # simple vote counting for self-consistency.
import re  # lightweight parsing for toy prompts and answers.
np.random.seed(0)  # reproducibility for sampled demonstrations and traces.

▶ What you'll see: the only tools are NumPy, Matplotlib, and Python's standard library.

### 1. Prompting as conditional computation

A prompt is not magic text; it is conditioning information. In a real language model we write $p(y \mid \mathrm{prompt}, x)$, meaning the answer distribution changes when the instruction, examples, or query changes. To make that visible, we will use two answer labels, start from zero-shot logits, and let prompt fields add simple logit shifts before softmax turns scores into probabilities.

In [ ]:
labels_w = np.array(["small", "large"])  # two possible outputs for the toy classifier.
zero_logits_w = np.array([1.0, 0.0])  # zero-shot scores before demonstrations or penalties.
exp_w = np.exp(zero_logits_w - np.max(zero_logits_w))  # stable exponentials for softmax.
probs_w = exp_w / exp_w.sum()  # convert logits into probabilities.
print("labels:", labels_w.tolist())
print("zero-shot probabilities:", np.round(probs_w, 3))
assert round(float(probs_w[0]), 3) == 0.731

▶ What you'll see: logits `[1, 0]` give probability 0.731 to the first label, matching the lesson's worked number.

In [ ]:
plt.figure(figsize=(4.2, 3))
plt.bar(labels_w, probs_w, color=["teal", "gray"])
plt.ylim(0, 1)
plt.ylabel("p(label | prompt, x)")
plt.title("1: zero-shot prompt-conditioned distribution")
plt.show()

▶ What you'll see: the `small` bar is taller because its logit is one unit higher.

*Why it's done this way:* logits are convenient because independent prompt effects add before normalization. Softmax then converts arbitrary real-valued scores into a probability distribution, so a prompt edit can be read as a controlled odds change rather than a vague wording trick.

### 2. Zero-shot versus few-shot demonstrations

Zero-shot prompting gives only instructions and a query. Few-shot prompting also supplies demonstrations, and those demonstrations bias the conditional distribution. Our toy rule says a matching example adds `+0.8` to the compatible label's logit, so one well-chosen example changes the first label's probability from $\sigma(1)=0.731$ to $\sigma(1.8)=0.858$.

In [ ]:
def sigmoid_w(z_w):
    return 1.0 / (1.0 + np.exp(-z_w))

base_margin_w = 1.0  # logit(small) - logit(large) in zero-shot mode.
demo_boost_w = 0.8  # one relevant demonstration favors the same label.
zero_prob_w = sigmoid_w(base_margin_w)
few_prob_w = sigmoid_w(base_margin_w + demo_boost_w)
print("zero-shot p(first option):", round(float(zero_prob_w), 3))
print("one-demo p(first option):", round(float(few_prob_w), 3))
assert round(float(few_prob_w), 3) == 0.858

▶ What you'll see: one demonstration raises the first option's probability from 0.731 to 0.858.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["zero-shot", "one-shot"], [zero_prob_w, few_prob_w], color=["gray", "seagreen"])
plt.ylim(0, 1)
plt.ylabel("probability of first option")
plt.title("2: demonstrations shift logits")
plt.show()

▶ What you'll see: the one-shot bar is higher because the demonstration adds log-odds evidence.

*Why it's done this way:* adding examples is a form of in-context statistical evidence. In the toy model, each matching example contributes an additive logit term; in real LMs, the Transformer attends to the example pattern, but the diagnostic question is the same: which token evidence changes the conditional distribution?

### 3. Prompt templates and parsing constraints

Prompting is also a formatting contract. If we ask for `label: ...`, then a parser can reliably extract the answer; if the model drifts into prose, the downstream system may fail. We will render a tiny template and apply a bad-format penalty of `-1.0`, whose odds multiplier is $e^{-1}=0.368$.

In [ ]:
def render_prompt_w(instruction_w, examples_w, query_w):
    lines_w = ["Instruction: " + instruction_w]
    for x_w, y_w in examples_w:
        lines_w.append("Input: " + x_w)
        lines_w.append("Output: label: " + y_w)
    lines_w.append("Input: " + query_w)
    lines_w.append("Output: label:")
    return "\n".join(lines_w)

prompt_w = render_prompt_w("Classify the integer as small or large.", [("2", "small"), ("9", "large")], "3")
print(prompt_w)

▶ What you'll see: instructions, demonstrations, query, and answer slot are concatenated into one context string.

In [ ]:
odds_multiplier_w = float(np.exp(-1.0))
print("bad-format odds multiplier:", round(odds_multiplier_w, 3))
assert round(odds_multiplier_w, 3) == 0.368

▶ What you'll see: a format penalty of `-1.0` cuts odds to about 36.8% of their previous value.

In [ ]:
plt.figure(figsize=(4.2, 3))
plt.bar(["well-formed", "bad format"], [1.0, odds_multiplier_w], color=["seagreen", "crimson"])
plt.ylim(0, 1.05)
plt.ylabel("relative odds")
plt.title("3: formatting contract penalty")
plt.show()

▶ What you'll see: the malformed answer path keeps only about 36.8% of the well-formed path's odds.

*Why it's done this way:* templates reduce ambiguity by making the target text easy to parse. A format penalty belongs in logit space because it should multiply odds, not subtract a fixed probability; that is why $e^{-1}$ is the natural diagnostic number.

### 4. Chain-of-thought as explicit intermediate state

Chain-of-thought asks the model to expose intermediate reasoning before the final answer. To avoid using a real model, we build a toy arithmetic reasoner with two modes: a brittle direct shortcut and a step-by-step trace that parses quantities, applies each operation, and then emits an answer. The goal is not to worship the text trace; it is to see why extra intermediate state can make the computation easier to check.

In [ ]:
task_w = "Mia has 3 apples, buys 4 apples, then gives away 2 apples."
nums_w = [int(n_w) for n_w in re.findall(r"\d+", task_w)]
print("parsed numbers:", nums_w)
assert nums_w == [3, 4, 2]

▶ What you'll see: the task is reduced to the quantities `[3, 4, 2]` before any answer is chosen.

In [ ]:
def direct_shortcut_w(text_w):
    nums = [int(n) for n in re.findall(r"\d+", text_w)]
    return nums[0] + nums[1] if "buys" in text_w else nums[0]

def cot_reasoner_w(text_w):
    nums = [int(n) for n in re.findall(r"\d+", text_w)]
    total = nums[0]
    trace = [("start", total)]
    if "buys" in text_w:
        total += nums[1]
        trace.append(("after buys", total))
    if "gives away" in text_w:
        total -= nums[2]
        trace.append(("after gives away", total))
    return total, trace

direct_w = direct_shortcut_w(task_w)
cot_answer_w, trace_w = cot_reasoner_w(task_w)
print("direct answer:", direct_w)
print("CoT trace:", trace_w, "answer:", cot_answer_w)
assert direct_w == 7 and cot_answer_w == 5

▶ What you'll see: the direct shortcut forgets the subtraction, while the trace reaches the correct answer 5.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot([step for step, value in trace_w], [value for step, value in trace_w], marker="o", color="purple")
plt.ylabel("running total")
plt.title("4: chain-of-thought exposes state")
plt.xticks(rotation=15)
plt.show()

▶ What you'll see: the running total goes 3 → 7 → 5, making the missing subtraction easy to inspect.

*Why it's done this way:* decomposing a computation into states changes the error surface. The direct answer must perform all operations in one opaque jump; the trace creates checkpoints where each local transition can be verified, which is why CoT often helps multi-step reasoning but can still be wrong if the intermediate state is ungrounded.

### 5. Self-consistency, context budget, and prompt cost

Prompting has resource tradeoffs. Longer prompts consume context, chain-of-thought consumes output tokens, and sampling multiple traces costs more but can improve reliability if errors are not perfectly correlated. For three independent traces with individual success rate 0.6, majority vote succeeds when exactly two or all three traces are correct: $3(0.6)^2(0.4)+(0.6)^3=0.648$.

In [ ]:
p_success_w = 0.6
majority_success_w = 3 * (p_success_w ** 2) * (1 - p_success_w) + p_success_w ** 3
context_window_w = 2048
prompt_tokens_w = 1200
remaining_w = context_window_w - prompt_tokens_w
print("majority success probability:", round(majority_success_w, 3))
print("tokens left for reasoning+answer:", remaining_w)
assert round(majority_success_w, 3) == 0.648
assert remaining_w == 848

▶ What you'll see: three traces lift success probability to 0.648, while a 1200-token prompt leaves 848 tokens.

In [ ]:
trace_votes_w = ["5", "5", "7"]
vote_counts_w = Counter(trace_votes_w)
final_w = vote_counts_w.most_common(1)[0][0]
print("trace votes:", trace_votes_w)
print("majority answer:", final_w)
plt.figure(figsize=(4, 3))
plt.bar(list(vote_counts_w.keys()), list(vote_counts_w.values()), color="darkorange")
plt.ylabel("votes")
plt.title("5: self-consistency vote")
plt.show()

▶ What you'll see: two traces vote for `5`, so majority voting rejects the lone `7`.

*Why it's done this way:* majority voting improves accuracy only when independent traces make partly independent mistakes; otherwise it repeats the same error. The context-budget arithmetic is equally important because every demonstration and reasoning token occupies finite space that could have held evidence or output.

## 🛠️ Setup

In [ ]:
import numpy as np  # Load NumPy for logits, probabilities, arrays, and simulations.
import matplotlib.pyplot as plt  # Load Matplotlib for compact debugging plots.
from collections import Counter  # Load Counter for majority votes in self-consistency examples.
import re  # Load regex parsing for tiny inline arithmetic prompts.
np.random.seed(0)  # Make all random choices reproducible.

def softmax(z):  # Convert logits to probabilities in a numerically stable way.
    z = np.asarray(z, dtype=float)  # Ensure vector arithmetic is predictable.
    shifted = z - np.max(z)  # Shift by the maximum so exponentials stay finite.
    exp = np.exp(shifted)  # Exponentiate shifted logits.
    return exp / exp.sum()  # Normalize into probabilities.

def sigmoid(z):  # Convert a two-class logit margin into a probability.
    return 1.0 / (1.0 + np.exp(-z))  # Use the logistic function.

def prompt_template(instruction, examples, query):  # Render a tiny few-shot prompt.
    lines = ["Instruction: " + instruction]  # Start with the task instruction.
    for x, y in examples:  # Add demonstrations in a consistent format.
        lines.append("Input: " + str(x))  # Add an input line.
        lines.append("Output: label: " + str(y))  # Add the target label line.
    lines.append("Input: " + str(query))  # Add the new query.
    lines.append("Output: label:")  # Leave a parseable answer slot.
    return "\n".join(lines)  # Join the prompt into one context string.

def parse_label(text):  # Extract label: VALUE from a generated string.
    match = re.search(r"label:\s*([A-Za-z0-9_+-]+)", text)  # Find the required label field.
    return None if match is None else match.group(1)  # Return None when formatting fails.

def toy_logits(x, examples=(), format_ok=True):  # Produce toy logits for labels small/large.
    x = int(x)  # Work with integer toy tasks.
    logits = np.array([1.0, 0.0]) if x < 5 else np.array([0.0, 1.0])  # Zero-shot rule: threshold at 5.
    target_label = "small" if x < 5 else "large"  # Identify the label compatible with the query.
    for ex_x, ex_y in examples:  # Let demonstrations add label evidence.
        if ex_y == target_label:  # Only same-label demonstrations support this query.
            logits[0 if x < 5 else 1] += 0.8  # Boost the compatible label.
    if not format_ok:  # Penalize malformed output.
        logits -= 1.0  # Apply a bad-format penalty to the whole answer path.
    return logits  # Return unnormalized scores.

def solve_apples(text, trace=False):  # Tiny rule-based reasoner for buy/give-away arithmetic.
    nums = [int(n) for n in re.findall(r"\d+", text)]  # Parse all numbers from the prompt.
    total = nums[0]  # Start from the initial amount.
    steps = [("start", total)]  # Record the first state.
    if "buys" in text or "gets" in text:  # Handle an addition operation.
        total += nums[1]  # Add the bought/gotten amount.
        steps.append(("after addition", total))  # Record the new state.
    if "gives away" in text or "loses" in text:  # Handle a subtraction operation.
        total -= nums[-1]  # Subtract the final amount.
        steps.append(("after subtraction", total))  # Record the new state.
    return (total, steps) if trace else total  # Return either answer only or answer plus trace.

def majority_vote(values):  # Choose the most common generated answer.
    return Counter(values).most_common(1)[0][0]  # Return the plurality winner.

## 🟢 Basics (warm-up)

### Basic 1 — Turn logits into probabilities

**Goal.** Convert prompt-conditioned logits into probabilities, because prompting changes scores before softmax makes them interpretable. We build it in 2 steps.

In [ ]:
logits_b1 = np.array([1.0, 0.0])  # Store zero-shot logits for two answer options.
labels_b1 = ["small", "large"]  # Name the two options.
print("logits:", logits_b1, "labels:", labels_b1)  # Inspect the unnormalized scores.

▶ What you'll see: the first option starts one logit unit ahead.

In [ ]:
probs_b1 = softmax(logits_b1)  # Normalize logits into probabilities.
print("probabilities:", np.round(probs_b1, 3))  # Inspect the prompt-conditioned distribution.
assert round(float(probs_b1[0]), 3) == 0.731  # Verify the lesson number.
plt.figure(figsize=(4, 3))  # Create a compact probability plot.
plt.bar(labels_b1, probs_b1, color="teal")  # Draw one bar per option.
plt.ylim(0, 1)  # Keep probability scale fixed.
plt.title("Basic 1: softmax over answer logits")  # Title the plot.
plt.ylabel("probability")  # Label the y-axis.
plt.show()  # Display the plot.

▶ What you'll see: the `small` bar is about 0.731, not 1.0, because softmax keeps uncertainty.

👀 Takeaway: prompt effects are easiest to reason about as logit shifts followed by softmax.

### Basic 2 — Render a zero-shot prompt

**Goal.** Build a prompt with no demonstrations, because zero-shot prompting relies only on instruction plus query. We build it in 2 steps.

In [ ]:
instruction_b2 = "Classify the integer as small (<5) or large (>=5)."  # Define the behavior request.
query_b2 = "3"  # Define the new input.
prompt_b2 = prompt_template(instruction_b2, [], query_b2)  # Render with no examples.
print(prompt_b2)  # Inspect the exact text context.

▶ What you'll see: the prompt contains an instruction, one input, and an empty label slot.

In [ ]:
logits_b2 = toy_logits(query_b2, examples=[])  # Score the query under the zero-shot toy rule.
probs_b2 = softmax(logits_b2)  # Convert scores to probabilities.
print("zero-shot probabilities:", np.round(probs_b2, 3))  # Inspect the model's answer distribution.
assert round(float(probs_b2[0]), 3) == 0.731  # Verify the canonical zero-shot probability.

▶ What you'll see: with no examples, the toy model uses only the instruction-like threshold rule.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a zero-shot probability plot.
plt.bar(["small", "large"], probs_b2, color=["teal", "gray"])  # Plot the two answer probabilities.
plt.ylim(0, 1)  # Keep a probability scale.
plt.title("Basic 2: zero-shot answer distribution")  # Title the chart.
plt.ylabel("probability")  # Label the y-axis.
plt.show()  # Display the plot.

▶ What you'll see: even without demonstrations, the prompt-conditioned rule favors `small` for query 3.

👀 Takeaway: zero-shot prompting conditions the model without adding task demonstrations.

### Basic 3 — Add one few-shot demonstration

**Goal.** Add one demonstration and measure the probability shift, because few-shot examples bias the conditional distribution. We build it in 2 steps.

In [ ]:
examples_b3 = [("2", "small")]  # Provide one demonstration matching the query side of the threshold.
prompt_b3 = prompt_template(instruction_b2, examples_b3, "3")  # Render a one-shot prompt.
print(prompt_b3)  # Inspect the demonstration format.

▶ What you'll see: the example shows the input-output pattern before the query.

In [ ]:
margin_b3 = 1.0 + 0.8  # Base margin plus one compatible example boost.
prob_b3 = sigmoid(margin_b3)  # Convert two-class margin to probability.
print("one-shot probability:", round(float(prob_b3), 3))  # Inspect the boosted probability.
assert round(float(prob_b3), 3) == 0.858  # Verify sigma(1.8).
plt.figure(figsize=(4, 3))  # Create a before-after plot.
plt.bar(["zero", "one-shot"], [sigmoid(1.0), prob_b3], color=["gray", "seagreen"])  # Compare probabilities.
plt.ylim(0, 1)  # Use probability scale.
plt.title("Basic 3: few-shot logit boost")  # Title the chart.
plt.show()  # Display the plot.

▶ What you'll see: the one-shot probability rises to about 0.858.

👀 Takeaway: few-shot examples act like extra conditioning evidence, not parameter updates.

### Basic 4 — Count prompt tokens approximately

**Goal.** Approximate context use with whitespace tokens, because every instruction and example consumes a finite window. We build it in 2 steps.

In [ ]:
prompt_b4 = prompt_template("Answer with label only.", [("2", "small"), ("9", "large")], "4")  # Build a short few-shot prompt.
tokens_b4 = prompt_b4.split()  # Use a simple whitespace token proxy for inspectability.
print("approx token count:", len(tokens_b4))  # Inspect prompt length.

▶ What you'll see: even a tiny prompt has multiple formatting and example tokens.

In [ ]:
window_b4 = 40  # Define a toy context window.
remaining_b4 = window_b4 - len(tokens_b4)  # Compute space left for reasoning and answer.
print("remaining toy-window tokens:", remaining_b4)  # Inspect the budget.
assert remaining_b4 > 0  # Verify the prompt fits.
plt.figure(figsize=(4, 3))  # Create a budget bar chart.
plt.bar(["used", "remaining"], [len(tokens_b4), remaining_b4], color=["orange", "teal"])  # Plot used versus left.
plt.title("Basic 4: prompt budget")  # Title the chart.
plt.ylabel("approx tokens")  # Label token counts.
plt.show()  # Display the plot.

▶ What you'll see: prompt tokens reduce how much space remains for the answer.

👀 Takeaway: demonstrations help, but they spend context that could hold reasoning or evidence.

### Basic 5 — Parse a structured answer

**Goal.** Extract a label from a formatted output, because prompting often succeeds only when downstream parsing is reliable. We build it in 2 steps.

In [ ]:
output_b5 = "label: small"  # Define a well-formed toy model output.
parsed_b5 = parse_label(output_b5)  # Extract the label field.
print("parsed label:", parsed_b5)  # Inspect the parsed value.
assert parsed_b5 == "small"  # Verify correct extraction.

▶ What you'll see: the parser finds `small` after the required `label:` prefix.

In [ ]:
bad_output_b5 = "The answer is small."  # Define an unstructured output.
bad_parsed_b5 = parse_label(bad_output_b5)  # Try the same parser.
print("bad parsed label:", bad_parsed_b5)  # Inspect the failure mode.
assert bad_parsed_b5 is None  # Verify malformed text is not silently accepted.

▶ What you'll see: the parser returns `None` when the contract is missing.

In [ ]:
parse_success_b5 = [parsed_b5 is not None, bad_parsed_b5 is not None]  # Mark whether each output was parseable.
plt.figure(figsize=(4.2, 3))  # Create a parse-success chart.
plt.bar(["label: small", "prose answer"], parse_success_b5, color=["seagreen", "crimson"])  # Plot parser outcomes.
plt.ylim(0, 1.05)  # Use a binary success scale.
plt.title("Basic 5: structured parsing succeeds")  # Title the chart.
plt.ylabel("parsed successfully")  # Label parser success.
plt.xticks(rotation=10)  # Rotate labels slightly.
plt.show()  # Display the plot.

▶ What you'll see: only the output that follows the `label:` contract is parseable.

👀 Takeaway: prompt format is part of the system, because later code may depend on exact fields.

### Basic 6 — Apply a bad-format penalty

**Goal.** Convert a format mistake into an odds penalty, because malformed outputs should be less likely or rejected. We build it in 2 steps.

In [ ]:
penalty_b6 = -1.0  # Define a logit penalty for not following the requested format.
multiplier_b6 = np.exp(penalty_b6)  # Convert logit penalty to an odds multiplier.
print("odds multiplier:", round(float(multiplier_b6), 3))  # Inspect e^-1.
assert round(float(multiplier_b6), 3) == 0.368  # Verify the lesson number.

▶ What you'll see: a `-1.0` logit penalty multiplies odds by 0.368.

In [ ]:
odds_before_b6 = 2.0  # Define prior odds for a correct formatted answer path.
odds_after_b6 = odds_before_b6 * multiplier_b6  # Apply the penalty in odds space.
print("odds before -> after:", round(odds_before_b6, 3), "->", round(float(odds_after_b6), 3))  # Inspect the effect.
plt.figure(figsize=(4, 3))  # Create an odds comparison plot.
plt.bar(["before", "after penalty"], [odds_before_b6, odds_after_b6], color=["gray", "crimson"])  # Plot odds before and after.
plt.title("Basic 6: format penalty changes odds")  # Title the chart.
plt.ylabel("odds")  # Label odds scale.
plt.show()  # Display the plot.

▶ What you'll see: the bad-format path becomes much less competitive.

👀 Takeaway: penalties are naturally additive in logits and multiplicative in odds.

### Basic 7 — Direct arithmetic shortcut

**Goal.** Show a brittle direct answer, because single-step prompting can skip a needed operation. We build it in 2 steps.

In [ ]:
task_b7 = "Mia has 3 apples, buys 4 apples, then gives away 2 apples."  # Define an inline arithmetic word problem.
nums_b7 = [int(n) for n in re.findall(r"\d+", task_b7)]  # Parse the quantities.
print("numbers:", nums_b7)  # Inspect the extracted quantities.
assert nums_b7 == [3, 4, 2]  # Verify parsing.

▶ What you'll see: all three quantities are present before reasoning.

In [ ]:
direct_b7 = nums_b7[0] + nums_b7[1]  # Use a brittle shortcut that ignores the give-away clause.
print("direct shortcut answer:", direct_b7)  # Inspect the wrong answer.
assert direct_b7 == 7  # Verify the shortcut's result.

▶ What you'll see: the shortcut answers 7 because it performs only the addition.

In [ ]:
true_b7 = nums_b7[0] + nums_b7[1] - nums_b7[2]  # Compute the full multi-step answer for comparison.
plt.figure(figsize=(4, 3))  # Create a direct-versus-correct plot.
plt.bar(["direct shortcut", "full reasoning"], [direct_b7, true_b7], color=["crimson", "teal"])  # Compare skipped versus complete computation.
plt.title("Basic 7: direct answer skips a step")  # Title the chart.
plt.ylabel("apples")  # Label answer values.
plt.show()  # Display the plot.

▶ What you'll see: the direct bar is too high because it omits the give-away subtraction.

👀 Takeaway: direct prompting can fail when the answer requires multiple dependent operations.

### Basic 8 — Trace chain-of-thought state

**Goal.** Compute the same task step by step, because a trace exposes intermediate state. We build it in 2 steps.

In [ ]:
answer_b8, steps_b8 = solve_apples("Mia has 3 apples, buys 4 apples, then gives away 2 apples.", trace=True)  # Run the toy CoT reasoner.
print("steps:", steps_b8)  # Inspect each state transition.
print("answer:", answer_b8)  # Inspect the final answer.
assert answer_b8 == 5  # Verify the correct result.

▶ What you'll see: the trace goes from start to addition to subtraction.

In [ ]:
plt.figure(figsize=(4.6, 3))  # Create a line plot for the trace.
plt.plot([s for s, v in steps_b8], [v for s, v in steps_b8], marker="o", color="purple")  # Draw state values over steps.
plt.xticks(rotation=15)  # Rotate labels for readability.
plt.ylabel("apples")  # Label running total.
plt.title("Basic 8: explicit reasoning state")  # Title the chart.
plt.show()  # Display the plot.

▶ What you'll see: the visible state lands on 5 after subtraction.

👀 Takeaway: chain-of-thought helps by creating checkable intermediate computation.

### Basic 9 — Majority vote three traces

**Goal.** Aggregate multiple generated answers, because self-consistency can reject one-off mistakes. We build it in 2 steps.

In [ ]:
answers_b9 = ["5", "5", "7"]  # Define three sampled trace answers.
counts_b9 = Counter(answers_b9)  # Count how many times each answer appears.
print("vote counts:", counts_b9)  # Inspect the votes.

▶ What you'll see: answer `5` has two votes and `7` has one.

In [ ]:
winner_b9 = majority_vote(answers_b9)  # Choose the most common answer.
print("majority winner:", winner_b9)  # Inspect the selected answer.
assert winner_b9 == "5"  # Verify majority vote.
plt.figure(figsize=(4, 3))  # Create a vote histogram.
plt.bar(list(counts_b9.keys()), list(counts_b9.values()), color="darkorange")  # Plot answer counts.
plt.title("Basic 9: self-consistency votes")  # Title the chart.
plt.ylabel("count")  # Label vote count.
plt.show()  # Display the plot.

▶ What you'll see: the majority bar for `5` is tallest.

👀 Takeaway: self-consistency is answer aggregation over sampled reasoning paths.

### Basic 10 — Compute context left

**Goal.** Subtract prompt length from a context window, because long prompts leave less room for reasoning and output. We build it in 2 steps.

In [ ]:
window_b10 = 2048  # Define the lesson context window.
prompt_len_b10 = 1200  # Define the prompt length from the lesson.
left_b10 = window_b10 - prompt_len_b10  # Compute available tokens.
print("tokens left:", left_b10)  # Inspect remaining space.
assert left_b10 == 848  # Verify the lesson number.

▶ What you'll see: a 1200-token prompt leaves 848 tokens.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a context budget chart.
plt.bar(["prompt", "left"], [prompt_len_b10, left_b10], color=["steelblue", "seagreen"])  # Plot used and remaining tokens.
plt.title("Basic 10: context-window budget")  # Title the chart.
plt.ylabel("tokens")  # Label token counts.
plt.show()  # Display the plot.

▶ What you'll see: more than half the window is already consumed by the prompt.

👀 Takeaway: prompt design is also resource allocation inside a fixed context window.

## 🟡 Easy

### Easy 1 — Compare zero-shot and few-shot on several queries

**Goal.** Score multiple integers with and without demonstrations, because few-shot prompting changes confidence across a batch of inputs. We build it in 3 steps.

In [ ]:
queries_e1 = np.array([1, 3, 6, 9])  # Define inline toy classification inputs.
examples_e1 = [("2", "small"), ("8", "large")]  # Provide balanced demonstrations.
print("queries:", queries_e1)  # Inspect the batch.
print("examples:", examples_e1)  # Inspect the demonstrations.

▶ What you'll see: two small-side and large-side tasks are ready to score.

In [ ]:
zero_probs_e1 = np.array([softmax(toy_logits(q, examples=[]))[0 if q < 5 else 1] for q in queries_e1])  # Confidence in correct labels without examples.
few_probs_e1 = np.array([softmax(toy_logits(q, examples=examples_e1))[0 if q < 5 else 1] for q in queries_e1])  # Confidence with examples.
print("zero-shot correct-label probs:", np.round(zero_probs_e1, 3))  # Inspect zero-shot confidence.
print("few-shot correct-label probs:", np.round(few_probs_e1, 3))  # Inspect few-shot confidence.
assert np.all(few_probs_e1 > zero_probs_e1)  # Verify demonstrations help in this toy setup.

▶ What you'll see: few-shot probabilities are higher for every query.

In [ ]:
x_e1 = np.arange(len(queries_e1))  # Create bar positions.
plt.figure(figsize=(5, 3))  # Create grouped bar chart.
plt.bar(x_e1 - 0.18, zero_probs_e1, width=0.36, label="zero-shot", color="gray")  # Plot zero-shot confidence.
plt.bar(x_e1 + 0.18, few_probs_e1, width=0.36, label="few-shot", color="teal")  # Plot few-shot confidence.
plt.xticks(x_e1, [str(q) for q in queries_e1])  # Label by query.
plt.ylim(0, 1)  # Use probability scale.
plt.title("Easy 1: demonstrations raise confidence")  # Title the plot.
plt.legend()  # Show labels.
plt.show()  # Display the plot.

▶ What you'll see: the few-shot bars sit above the zero-shot bars.

👀 Takeaway: examples can shift the conditional distribution without changing model parameters.

### Easy 2 — Detect label imbalance in demonstrations

**Goal.** Count demonstration labels, because few-shot examples with imbalance can bias outputs. We build it in 3 steps.

In [ ]:
examples_e2 = [("1", "small"), ("2", "small"), ("3", "small"), ("9", "large")]  # Create imbalanced examples.
labels_e2 = [y for x, y in examples_e2]  # Extract labels.
counts_e2 = Counter(labels_e2)  # Count label frequencies.
print("label counts:", counts_e2)  # Inspect imbalance.

▶ What you'll see: `small` appears three times while `large` appears once.

In [ ]:
imbalance_e2 = counts_e2["small"] / len(examples_e2)  # Compute small-label share.
print("small share:", round(imbalance_e2, 3))  # Inspect the imbalance proportion.
assert round(imbalance_e2, 3) == 0.75  # Verify concrete number.

▶ What you'll see: 75% of the demonstrations have the same label.

In [ ]:
plt.figure(figsize=(4, 3))  # Create label-count plot.
plt.bar(list(counts_e2.keys()), list(counts_e2.values()), color=["seagreen", "orange"])  # Plot counts.
plt.title("Easy 2: few-shot label balance")  # Title the chart.
plt.ylabel("demonstrations")  # Label count scale.
plt.show()  # Display the plot.

▶ What you'll see: the imbalanced label has a visibly taller bar.

👀 Takeaway: demonstrations are conditioning evidence, so their label distribution matters.

### Easy 3 — Check chain-of-thought against direct answers

**Goal.** Evaluate direct and traced reasoning on a small task set, because CoT gains appear on multi-step cases. We build it in 4 steps.

In [ ]:
tasks_e3 = [
    "Mia has 3 apples, buys 4 apples, then gives away 2 apples.",
    "Noah has 5 apples, buys 2 apples, then gives away 1 apples.",
    "Lia has 6 apples, buys 3 apples, then gives away 4 apples."
]  # Define inline arithmetic tasks.
truth_e3 = np.array([5, 6, 5])  # Define correct answers.
print("number of tasks:", len(tasks_e3))  # Inspect task count.

▶ What you'll see: three small multi-step tasks are ready.

In [ ]:
direct_e3 = np.array([[int(n) for n in re.findall(r"\d+", t)][0] + [int(n) for n in re.findall(r"\d+", t)][1] for t in tasks_e3])  # Direct shortcut ignores subtraction.
cot_e3 = np.array([solve_apples(t) for t in tasks_e3])  # Traced reasoner handles addition and subtraction.
print("direct:", direct_e3, "CoT:", cot_e3, "truth:", truth_e3)  # Inspect predictions.

▶ What you'll see: direct answers are too high, while CoT answers match truth.

In [ ]:
acc_direct_e3 = float(np.mean(direct_e3 == truth_e3))  # Compute direct accuracy.
acc_cot_e3 = float(np.mean(cot_e3 == truth_e3))  # Compute CoT accuracy.
print("direct accuracy:", acc_direct_e3, "CoT accuracy:", acc_cot_e3)  # Inspect the gain.
assert acc_direct_e3 == 0.0 and acc_cot_e3 == 1.0  # Verify the toy CoT gain.

▶ What you'll see: direct accuracy is 0, while traced reasoning is 1 on this set.

In [ ]:
plt.figure(figsize=(4, 3))  # Create accuracy comparison.
plt.bar(["direct", "CoT"], [acc_direct_e3, acc_cot_e3], color=["crimson", "teal"])  # Plot accuracies.
plt.ylim(0, 1.05)  # Use accuracy scale.
plt.title("Easy 3: CoT helps multi-step tasks")  # Title the chart.
plt.ylabel("accuracy")  # Label y-axis.
plt.show()  # Display the plot.

▶ What you'll see: the CoT bar reaches 1.0 while direct stays at 0.

👀 Takeaway: chain-of-thought helps when the answer depends on ordered intermediate operations.

### Easy 4 — Estimate majority-vote reliability

**Goal.** Compute majority success for different per-trace success rates, because self-consistency depends on individual trace quality. We build it in 3 steps.

In [ ]:
p_grid_e4 = np.array([0.4, 0.5, 0.6, 0.7, 0.8])  # Define individual trace success rates.
maj_e4 = 3 * (p_grid_e4 ** 2) * (1 - p_grid_e4) + p_grid_e4 ** 3  # Probability at least two of three are correct.
print("single-trace p:", p_grid_e4)  # Inspect input rates.
print("majority success:", np.round(maj_e4, 3))  # Inspect majority-vote rates.
assert round(float(maj_e4[2]), 3) == 0.648  # Verify p=0.6 lesson number.

▶ What you'll see: majority voting helps when individual traces are better than chance.

In [ ]:
improvement_e4 = maj_e4 - p_grid_e4  # Compute the gain over one trace.
print("improvement:", np.round(improvement_e4, 3))  # Inspect where voting helps or hurts.

▶ What you'll see: improvement is negative below 0.5, zero at 0.5, and positive above 0.5.

In [ ]:
plt.figure(figsize=(5, 3))  # Create reliability curve.
plt.plot(p_grid_e4, p_grid_e4, marker="o", label="one trace", color="gray")  # Plot single-trace baseline.
plt.plot(p_grid_e4, maj_e4, marker="o", label="3-trace majority", color="purple")  # Plot majority success.
plt.title("Easy 4: majority vote needs decent traces")  # Title the chart.
plt.xlabel("single-trace success")  # Label x-axis.
plt.ylabel("success probability")  # Label y-axis.
plt.legend()  # Show labels.
plt.show()  # Display the plot.

▶ What you'll see: the majority curve is above the diagonal only after 0.5.

👀 Takeaway: self-consistency is not free accuracy; it amplifies the quality of its samples.

### Easy 5 — Score prompt variants with accuracy and cost

**Goal.** Compare prompt variants on a toy utility, because better behavior must be weighed against token cost. We build it in 4 steps.

In [ ]:
names_e5 = np.array(["zero", "few-shot", "few+CoT"])  # Define prompt variants.
accuracy_e5 = np.array([0.70, 0.82, 0.90])  # Toy validation accuracies.
tokens_e5 = np.array([40, 120, 260])  # Approximate prompt+answer costs.
print("variants:", names_e5.tolist())  # Inspect variant names.

▶ What you'll see: accuracy rises as prompts become longer.

In [ ]:
cost_penalty_e5 = 0.0005 * tokens_e5  # Convert token count into a small utility penalty.
utility_e5 = accuracy_e5 - cost_penalty_e5  # Compute accuracy minus cost.
print("utility:", np.round(utility_e5, 3))  # Inspect cost-adjusted value.

▶ What you'll see: the best raw accuracy is not automatically the best utility.

In [ ]:
best_idx_e5 = int(np.argmax(utility_e5))  # Select the highest utility prompt.
print("best utility variant:", names_e5[best_idx_e5])  # Inspect selected variant.
assert names_e5[best_idx_e5] == "few+CoT"  # Verify this toy tradeoff still favors CoT.

▶ What you'll see: `few+CoT` wins under this specific cost penalty.

In [ ]:
plt.figure(figsize=(5, 3))  # Create utility bar chart.
plt.bar(names_e5, utility_e5, color=["gray", "teal", "purple"])  # Plot utilities.
plt.title("Easy 5: prompt quality minus token cost")  # Title the chart.
plt.ylabel("utility")  # Label y-axis.
plt.show()  # Display the plot.

▶ What you'll see: utility summarizes both correctness and cost.

👀 Takeaway: prompt selection should be evaluated, not chosen by intuition alone.

## 🔴 Advanced

### Advanced 1 — Optimize demonstration selection

**Goal.** Choose demonstrations that improve validation confidence, because few-shot examples are a small training set inside the prompt. We build it in 4 steps.

In [ ]:
candidate_examples_a1 = [("1", "small"), ("2", "small"), ("8", "large"), ("9", "large")]  # Candidate demonstrations.
validation_a1 = np.array([3, 4, 6, 7])  # Validation inputs around the threshold.
print("candidates:", candidate_examples_a1)  # Inspect demo pool.

▶ What you'll see: the pool has two examples for each label.

In [ ]:
sets_a1 = [candidate_examples_a1[:2], candidate_examples_a1[2:], [candidate_examples_a1[0], candidate_examples_a1[2]]]  # Try small-only, large-only, balanced.
set_names_a1 = ["small-only", "large-only", "balanced"]  # Name each prompt design.
scores_a1 = []  # Store mean correct-label confidence.
for demo_set_a1 in sets_a1:  # Evaluate each set.
    confs_a1 = [softmax(toy_logits(q, examples=demo_set_a1))[0 if q < 5 else 1] for q in validation_a1]  # Correct-label probabilities.
    scores_a1.append(np.mean(confs_a1))  # Store average confidence.
print("mean confidence:", dict(zip(set_names_a1, np.round(scores_a1, 3))))  # Inspect scores.

▶ What you'll see: balanced demonstrations produce the strongest average confidence.

In [ ]:
best_a1 = set_names_a1[int(np.argmax(scores_a1))]  # Select the best demo set.
print("best demo set:", best_a1)  # Inspect the selection.
assert best_a1 == "balanced"  # Verify balanced examples win in this toy setup.

▶ What you'll see: the balanced set is selected.

In [ ]:
plt.figure(figsize=(5, 3))  # Create demo-selection plot.
plt.bar(set_names_a1, scores_a1, color=["orange", "steelblue", "seagreen"])  # Plot validation confidence.
plt.title("Advanced 1: choosing demonstrations")  # Title the chart.
plt.ylabel("mean correct-label probability")  # Label y-axis.
plt.xticks(rotation=10)  # Rotate labels.
plt.show()  # Display the plot.

▶ What you'll see: validation scoring exposes which examples are useful.

👀 Takeaway: few-shot examples should be selected with validation behavior in mind, not just copied at random.

### Advanced 2 — Simulate correlated chain-of-thought errors

**Goal.** Compare independent and correlated trace errors, because self-consistency fails when samples repeat the same mistake. We build it in 4 steps.

In [ ]:
rng_a2 = np.random.default_rng(2)  # Create reproducible simulation randomness.
n_trials_a2 = 2000  # Define number of simulated questions.
p_a2 = 0.6  # Define individual trace success probability.
print("trials:", n_trials_a2, "single trace p:", p_a2)  # Inspect simulation settings.

▶ What you'll see: the simulation uses the lesson's 0.6 trace success rate.

In [ ]:
independent_a2 = rng_a2.random((n_trials_a2, 3)) < p_a2  # Three independent trace correctness flags.
maj_ind_a2 = np.mean(np.sum(independent_a2, axis=1) >= 2)  # Majority succeeds if at least two are correct.
print("independent majority success:", round(float(maj_ind_a2), 3))  # Inspect empirical success.

▶ What you'll see: independent majority success is close to the theoretical 0.648.

In [ ]:
shared_correct_a2 = rng_a2.random(n_trials_a2) < p_a2  # One shared latent outcome per question.
correlated_a2 = np.repeat(shared_correct_a2[:, None], 3, axis=1)  # All traces repeat that outcome.
maj_corr_a2 = np.mean(np.sum(correlated_a2, axis=1) >= 2)  # Majority vote under perfect correlation.
print("correlated majority success:", round(float(maj_corr_a2), 3))  # Inspect correlated success.
assert abs(maj_corr_a2 - p_a2) < 0.04  # Verify correlation removes most voting gain.

▶ What you'll see: correlated voting stays near 0.6 instead of improving to 0.648.

In [ ]:
plt.figure(figsize=(4, 3))  # Create comparison plot.
plt.bar(["independent", "correlated"], [maj_ind_a2, maj_corr_a2], color=["teal", "crimson"])  # Plot success rates.
plt.axhline(p_a2, color="gray", linestyle="--", label="one trace")  # Show one-trace baseline.
plt.title("Advanced 2: correlated errors reduce voting gains")  # Title the chart.
plt.ylabel("majority success")  # Label y-axis.
plt.legend()  # Show baseline label.
plt.show()  # Display the plot.

▶ What you'll see: independent samples help more than perfectly correlated samples.

👀 Takeaway: self-consistency assumes diversity; identical reasoning failures cannot be voted away.

### Advanced 3 — Build a parse-and-repair loop

**Goal.** Retry malformed outputs with a stricter prompt, because format failures are often cheaper to repair than to accept. We build it in 4 steps.

In [ ]:
outputs_a3 = ["The answer is small.", "label: small"]  # Simulate first malformed output, then repaired output.
parsed_a3 = [parse_label(o) for o in outputs_a3]  # Parse both attempts.
print("parsed attempts:", parsed_a3)  # Inspect parser results.

▶ What you'll see: the first attempt fails, while the second returns `small`.

In [ ]:
attempts_a3 = 0  # Count attempts used.
final_a3 = None  # Store final parsed answer.
for out_a3 in outputs_a3:  # Iterate through simulated attempts.
    attempts_a3 += 1  # Increment attempt count.
    final_a3 = parse_label(out_a3)  # Try to parse the output.
    if final_a3 is not None:  # Stop once the format is valid.
        break  # End the repair loop.
print("attempts used:", attempts_a3, "final:", final_a3)  # Inspect loop result.
assert attempts_a3 == 2 and final_a3 == "small"  # Verify repair succeeds on second try.

▶ What you'll see: the repair loop stops after the first valid structured answer.

In [ ]:
cost_per_attempt_a3 = 50  # Define a toy token cost per generation.
total_cost_a3 = attempts_a3 * cost_per_attempt_a3  # Compute retry cost.
print("total token cost:", total_cost_a3)  # Inspect repair cost.

▶ What you'll see: two attempts cost twice one attempt.

In [ ]:
plt.figure(figsize=(4, 3))  # Create attempts plot.
plt.bar(["attempts", "cost/50"], [attempts_a3, total_cost_a3 / 50], color=["purple", "gray"])  # Plot equivalent counts.
plt.title("Advanced 3: repair loop cost")  # Title the chart.
plt.ylabel("count")  # Label y-axis.
plt.show()  # Display the plot.

▶ What you'll see: repair works, but it spends extra generation budget.

👀 Takeaway: output contracts should include validation and a stopping rule for retries.

### Advanced 4 — Allocate a fixed context budget

**Goal.** Choose how many examples fit alongside reasoning space, because prompt length and answer length compete inside one window. We build it in 4 steps.

In [ ]:
window_a4 = 256  # Define a toy context window.
instruction_a4 = 35  # Reserve instruction tokens.
query_a4 = 20  # Reserve query tokens.
reasoning_a4 = 80  # Reserve expected reasoning and answer tokens.
example_cost_a4 = 30  # Define tokens per demonstration.
print("window:", window_a4, "fixed tokens:", instruction_a4 + query_a4 + reasoning_a4)  # Inspect fixed cost.

▶ What you'll see: fixed prompt parts already use 135 tokens.

In [ ]:
available_for_examples_a4 = window_a4 - instruction_a4 - query_a4 - reasoning_a4  # Compute demo budget.
max_examples_a4 = available_for_examples_a4 // example_cost_a4  # Fit whole demonstrations only.
print("available for examples:", available_for_examples_a4)  # Inspect remaining demo space.
print("max examples:", max_examples_a4)  # Inspect how many fit.
assert max_examples_a4 == 4  # Verify the budget calculation.

▶ What you'll see: four full demonstrations fit with this budget.

In [ ]:
counts_a4 = np.arange(0, 7)  # Try 0 through 6 examples.
used_a4 = instruction_a4 + query_a4 + reasoning_a4 + counts_a4 * example_cost_a4  # Compute total tokens used.
fits_a4 = used_a4 <= window_a4  # Check which counts fit.
print("fits:", dict(zip(counts_a4, fits_a4)))  # Inspect feasibility by example count.

▶ What you'll see: counts above four exceed the window.

In [ ]:
plt.figure(figsize=(5, 3))  # Create context allocation plot.
plt.bar(counts_a4, used_a4, color=np.where(fits_a4, "teal", "crimson"))  # Green fits, red overflows.
plt.axhline(window_a4, color="black", linestyle="--", label="window")  # Show max context.
plt.title("Advanced 4: examples compete with reasoning space")  # Title the chart.
plt.xlabel("number of demonstrations")  # Label x-axis.
plt.ylabel("tokens used")  # Label y-axis.
plt.legend()  # Show window label.
plt.show()  # Display the plot.

▶ What you'll see: the bars turn infeasible once examples consume too much space.

👀 Takeaway: context engineering is a constrained allocation problem, not just adding more examples.

### Advanced 5 — Evaluate prompt robustness under distribution shift

**Goal.** Compare prompt variants on in-distribution and shifted tasks, because prompt tricks are distribution shifts rather than guaranteed algorithms. We build it in 4 steps.

In [ ]:
variants_a5 = np.array(["zero", "few-shot", "few+CoT"])  # Define prompt variants.
in_dist_a5 = np.array([0.72, 0.86, 0.92])  # Toy in-distribution accuracies.
shifted_a5 = np.array([0.68, 0.70, 0.82])  # Toy shifted accuracies.
print("in-distribution:", in_dist_a5)  # Inspect baseline results.
print("shifted:", shifted_a5)  # Inspect shifted results.

▶ What you'll see: every variant loses accuracy under shift, but not equally.

In [ ]:
drop_a5 = in_dist_a5 - shifted_a5  # Compute robustness gap.
print("accuracy drop:", np.round(drop_a5, 3))  # Inspect degradation.
assert round(float(drop_a5[1]), 3) == 0.16  # Verify few-shot drop number.

▶ What you'll see: few-shot loses 0.16 in this toy setup.

In [ ]:
robust_idx_a5 = int(np.argmin(drop_a5))  # Select smallest degradation.
print("most robust variant:", variants_a5[robust_idx_a5])  # Inspect robustness winner.
assert variants_a5[robust_idx_a5] == "zero"  # Verify zero-shot has smallest drop here.

▶ What you'll see: zero-shot is most stable by drop, even though it is not most accurate.

In [ ]:
x_a5 = np.arange(len(variants_a5))  # Create grouped bar positions.
plt.figure(figsize=(5, 3))  # Create robustness chart.
plt.bar(x_a5 - 0.18, in_dist_a5, width=0.36, label="in-dist", color="teal")  # Plot in-distribution accuracy.
plt.bar(x_a5 + 0.18, shifted_a5, width=0.36, label="shifted", color="orange")  # Plot shifted accuracy.
plt.xticks(x_a5, variants_a5)  # Label variants.
plt.ylim(0, 1)  # Use accuracy scale.
plt.title("Advanced 5: prompt robustness check")  # Title the chart.
plt.ylabel("accuracy")  # Label y-axis.
plt.legend()  # Show labels.
plt.show()  # Display the plot.

▶ What you'll see: the shifted bars are lower, showing distribution sensitivity.

👀 Takeaway: evaluate prompting under the behavior you need, not only on examples that resemble the prompt.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Prompting is programming the conditional distribution with words and examples.

Prompting changes conditioning text, not model weights. This notebook uses small logits to model instructions, demonstrations, format penalties, and chain-of-thought voting without any LLM call. Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 91419
rng = np.random.default_rng(SEED)
random.seed(SEED)


def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-float(x)))


def softmax(values):
    arr = np.asarray(values, dtype=float)
    shifted = arr - np.max(arr)
    weights = np.exp(shifted)
    return weights / weights.sum()


def normalize_rows(matrix):
    arr = np.asarray(matrix, dtype=float)
    return arr / arr.sum(axis=1, keepdims=True)


def kl_divergence(policy, reference):
    policy = np.asarray(policy, dtype=float)
    reference = np.asarray(reference, dtype=float)
    safe_policy = np.clip(policy, 1e-9, 1.0)
    safe_reference = np.clip(reference, 1e-9, 1.0)
    return float(np.sum(safe_policy * (np.log(safe_policy) - np.log(safe_reference))))


def make_f8_ladder(topic):
    if topic == "rlhf":
        return build_rlhf_ladder()
    if topic == "dpo":
        return build_dpo_ladder()
    if topic == "constitutional":
        return build_constitutional_ladder()
    if topic == "icl":
        return build_icl_ladder()
    if topic == "prompting":
        return build_prompting_ladder()
    if topic == "reasoning":
        return build_reasoning_ladder()
    raise ValueError(topic)


def build_rlhf_ladder():
    action_names = ["concise_helpful", "verbose_loophole", "refusal"]
    return [
        {
            "name": "D1 one prompt/action",
            "actions": action_names,
            "reference": np.array([[0.62, 0.25, 0.13]]),
            "policy": np.array([[0.55, 0.32, 0.13]]),
            "reward_model": np.array([[2.0, 1.0, 0.2]]),
            "human_reward": np.array([[2.0, 1.0, 0.2]]),
            "advantages": np.array([[0.5, -0.2, -0.4]]),
            "ratios": np.array([[1.2, 0.8, 1.0]]),
            "beta": 0.10,
        },
        {
            "name": "D2 few-shot preference-policy set",
            "actions": action_names,
            "reference": normalize_rows([[0.55, 0.30, 0.15], [0.50, 0.25, 0.25]]),
            "policy": normalize_rows([[0.62, 0.28, 0.10], [0.58, 0.30, 0.12]]),
            "reward_model": np.array([[2.1, 1.4, 0.2], [1.8, 1.2, 0.5]]),
            "human_reward": np.array([[2.0, 1.0, 0.2], [1.7, 1.0, 0.7]]),
            "advantages": np.array([[0.6, -0.1, -0.5], [0.4, 0.0, -0.3]]),
            "ratios": np.array([[1.2, 0.9, 0.7], [1.1, 1.0, 0.8]]),
            "beta": 0.12,
        },
        {
            "name": "D3 distractor reward loopholes",
            "actions": action_names,
            "reference": normalize_rows([[0.48, 0.37, 0.15], [0.45, 0.35, 0.20], [0.50, 0.30, 0.20]]),
            "policy": normalize_rows([[0.44, 0.48, 0.08], [0.42, 0.45, 0.13], [0.54, 0.34, 0.12]]),
            "reward_model": np.array([[2.0, 2.6, 0.2], [1.9, 2.4, 0.3], [2.1, 1.8, 0.4]]),
            "human_reward": np.array([[2.0, 1.1, 0.2], [1.8, 1.0, 0.4], [2.0, 1.2, 0.6]]),
            "advantages": np.array([[0.4, 0.7, -0.4], [0.3, 0.6, -0.3], [0.5, 0.2, -0.2]]),
            "ratios": np.array([[1.0, 1.4, 0.7], [0.9, 1.3, 0.8], [1.1, 1.1, 0.8]]),
            "beta": 0.08,
        },
        {
            "name": "D4 real-style instruction/reward set",
            "actions": action_names,
            "reference": normalize_rows([[0.50, 0.30, 0.20], [0.45, 0.25, 0.30], [0.40, 0.35, 0.25], [0.55, 0.25, 0.20]]),
            "policy": normalize_rows([[0.63, 0.27, 0.10], [0.58, 0.27, 0.15], [0.55, 0.33, 0.12], [0.66, 0.23, 0.11]]),
            "reward_model": np.array([[2.2, 1.4, 0.5], [2.0, 1.3, 0.8], [1.9, 1.7, 0.4], [2.3, 1.2, 0.3]]),
            "human_reward": np.array([[2.1, 1.1, 0.5], [1.9, 1.0, 0.9], [1.8, 1.1, 0.5], [2.2, 1.0, 0.4]]),
            "advantages": np.array([[0.7, 0.0, -0.4], [0.5, -0.1, -0.2], [0.4, 0.1, -0.4], [0.8, -0.2, -0.5]]),
            "ratios": np.array([[1.2, 0.9, 0.7], [1.2, 1.0, 0.8], [1.1, 1.1, 0.8], [1.2, 0.8, 0.7]]),
            "beta": 0.12,
        },
        {
            "name": "D5 longer context with KL drift",
            "actions": action_names,
            "reference": normalize_rows([[0.52, 0.28, 0.20], [0.49, 0.31, 0.20], [0.46, 0.34, 0.20], [0.50, 0.27, 0.23], [0.47, 0.32, 0.21], [0.51, 0.29, 0.20]]),
            "policy": normalize_rows([[0.40, 0.55, 0.05], [0.42, 0.50, 0.08], [0.44, 0.47, 0.09], [0.61, 0.30, 0.09], [0.46, 0.45, 0.09], [0.63, 0.28, 0.09]]),
            "reward_model": np.array([[2.1, 3.0, 0.4], [2.0, 2.8, 0.4], [1.9, 2.7, 0.3], [2.2, 1.5, 0.5], [2.0, 2.5, 0.4], [2.3, 1.4, 0.3]]),
            "human_reward": np.array([[2.0, 1.0, 0.5], [1.9, 0.9, 0.5], [1.8, 1.0, 0.4], [2.1, 1.1, 0.5], [1.9, 0.8, 0.5], [2.2, 1.0, 0.4]]),
            "advantages": np.array([[0.5, 1.0, -0.3], [0.4, 0.9, -0.3], [0.3, 0.8, -0.4], [0.7, 0.1, -0.4], [0.4, 0.7, -0.3], [0.8, 0.0, -0.5]]),
            "ratios": np.array([[0.9, 1.6, 0.5], [0.9, 1.5, 0.6], [1.0, 1.4, 0.6], [1.2, 1.0, 0.6], [1.0, 1.4, 0.6], [1.2, 0.9, 0.6]]),
            "beta": 0.04,
        },
    ]


def build_dpo_ladder():
    return [
        {
            "name": "D1 one prompt preference",
            "policy_chosen": np.array([-1.0]),
            "policy_rejected": np.array([-2.0]),
            "reference_chosen": np.array([-1.5]),
            "reference_rejected": np.array([-2.2]),
            "labels": np.array([1]),
            "beta": 2.0,
        },
        {
            "name": "D2 few-shot preference set",
            "policy_chosen": np.array([-0.8, -1.1, -1.0]),
            "policy_rejected": np.array([-1.8, -1.7, -2.0]),
            "reference_chosen": np.array([-1.2, -1.4, -1.5]),
            "reference_rejected": np.array([-1.9, -1.9, -2.2]),
            "labels": np.array([1, 1, 1]),
            "beta": 2.0,
        },
        {
            "name": "D3 label noise and distractors",
            "policy_chosen": np.array([-0.7, -1.8, -0.9, -1.3]),
            "policy_rejected": np.array([-1.6, -1.4, -1.7, -1.2]),
            "reference_chosen": np.array([-1.1, -1.5, -1.4, -1.4]),
            "reference_rejected": np.array([-1.8, -1.8, -2.0, -1.6]),
            "labels": np.array([1, 0, 1, 0]),
            "beta": 2.0,
        },
        {
            "name": "D4 real-style pair corpus",
            "policy_chosen": np.array([-0.7, -0.9, -1.1, -0.8, -1.0]),
            "policy_rejected": np.array([-1.8, -1.5, -1.6, -1.7, -1.9]),
            "reference_chosen": np.array([-1.2, -1.2, -1.4, -1.3, -1.5]),
            "reference_rejected": np.array([-1.9, -1.7, -1.9, -1.8, -2.1]),
            "labels": np.array([1, 1, 1, 1, 1]),
            "beta": 1.5,
        },
        {
            "name": "D5 longer context with saturated beta",
            "policy_chosen": np.array([-0.5, -0.6, -2.2, -0.7, -2.0, -0.9]),
            "policy_rejected": np.array([-2.0, -1.9, -1.1, -1.8, -1.0, -1.6]),
            "reference_chosen": np.array([-1.2, -1.3, -1.8, -1.4, -1.7, -1.5]),
            "reference_rejected": np.array([-2.1, -2.0, -1.7, -2.0, -1.6, -1.9]),
            "labels": np.array([1, 1, 0, 1, 0, 1]),
            "beta": 6.0,
        },
    ]


def build_constitutional_ladder():
    principles = ["harmlessness", "honesty", "helpfulness"]
    return [
        {
            "name": "D1 one prompt/principle",
            "principles": principles[:2],
            "weights": np.array([2.0, 1.0]),
            "violations": np.array([[0.3, 0.8]]),
            "revised": np.array([[0.2, 0.2]]),
            "task_success": np.array([0.90]),
        },
        {
            "name": "D2 few-shot critique set",
            "principles": principles,
            "weights": np.array([2.0, 1.0, 1.5]),
            "violations": np.array([[0.4, 0.5, 0.2], [0.2, 0.8, 0.3]]),
            "revised": np.array([[0.2, 0.2, 0.2], [0.2, 0.3, 0.2]]),
            "task_success": np.array([0.88, 0.86]),
        },
        {
            "name": "D3 conflicting principles and distractors",
            "principles": principles,
            "weights": np.array([2.5, 1.0, 0.8]),
            "violations": np.array([[0.6, 0.3, 0.7], [0.5, 0.6, 0.6], [0.2, 0.4, 0.8]]),
            "revised": np.array([[0.2, 0.2, 0.5], [0.2, 0.3, 0.5], [0.1, 0.2, 0.6]]),
            "task_success": np.array([0.70, 0.72, 0.68]),
        },
        {
            "name": "D4 real-style policy examples",
            "principles": principles,
            "weights": np.array([2.0, 1.3, 1.4]),
            "violations": np.array([[0.5, 0.2, 0.3], [0.2, 0.7, 0.4], [0.4, 0.5, 0.5], [0.3, 0.2, 0.8]]),
            "revised": np.array([[0.2, 0.2, 0.3], [0.2, 0.2, 0.3], [0.2, 0.2, 0.4], [0.2, 0.2, 0.5]]),
            "task_success": np.array([0.86, 0.88, 0.84, 0.82]),
        },
        {
            "name": "D5 longer context with vague scores",
            "principles": principles,
            "weights": np.array([3.0, 1.0, 0.3]),
            "violations": np.array([[0.7, 0.3, 0.8], [0.6, 0.4, 0.9], [0.5, 0.5, 0.8], [0.4, 0.6, 0.9], [0.7, 0.2, 0.7]]),
            "revised": np.array([[0.1, 0.2, 0.7], [0.1, 0.3, 0.8], [0.1, 0.3, 0.7], [0.1, 0.4, 0.8], [0.1, 0.2, 0.6]]),
            "task_success": np.array([0.46, 0.42, 0.48, 0.40, 0.44]),
        },
    ]


def build_icl_ladder():
    return [
        {
            "name": "D1 one prompt",
            "scores": np.array([[2.0, 1.0, 0.0]]),
            "labels": np.array([[1.0, 0.0, 1.0]]),
            "targets": np.array([1]),
            "token_budget": 8,
            "demo_tokens": 6,
            "recency_bonus": 0.5,
        },
        {
            "name": "D2 few-shot set",
            "scores": np.array([[2.2, 1.0, 0.1], [0.4, 1.8, 0.2], [1.7, 0.8, 0.5]]),
            "labels": np.array([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0]]),
            "targets": np.array([1, 1, 1]),
            "token_budget": 16,
            "demo_tokens": 8,
            "recency_bonus": 0.4,
        },
        {
            "name": "D3 distractors and order flips",
            "scores": np.array([[2.0, 1.9, 0.1, 0.0], [0.2, 1.6, 1.5, 0.1], [1.4, 0.3, 1.3, 0.2], [0.1, 0.2, 1.7, 1.6]]),
            "labels": np.array([[1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 0.0, 0.0], [1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 0.0, 0.0]]),
            "targets": np.array([1, 1, 1, 1]),
            "token_budget": 24,
            "demo_tokens": 16,
            "recency_bonus": 0.6,
        },
        {
            "name": "D4 real text-label examples",
            "scores": np.array([[2.3, 1.2, 0.6, 0.1], [0.5, 2.1, 1.0, 0.2], [1.9, 0.8, 0.7, 0.4], [0.4, 1.7, 1.4, 0.3], [2.1, 0.7, 0.8, 0.2]]),
            "labels": np.array([[1.0, 0.0, 1.0, 0.0], [0.0, 1.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [1.0, 0.0, 1.0, 0.0]]),
            "targets": np.array([1, 1, 1, 1, 1]),
            "token_budget": 40,
            "demo_tokens": 24,
            "recency_bonus": 0.3,
        },
        {
            "name": "D5 longer context with diluted attention",
            "scores": np.array([[2.0, 1.9, 1.8, 1.7, 0.2, 0.1], [0.3, 1.8, 1.7, 1.6, 0.2, 0.1], [1.7, 1.6, 1.5, 1.4, 0.3, 0.2], [0.2, 1.7, 1.6, 1.5, 0.3, 0.2], [1.8, 1.7, 1.6, 1.5, 0.3, 0.2], [0.2, 1.6, 1.5, 1.4, 0.3, 0.2]]),
            "labels": np.array([[1.0, 0.0, 0.0, 0.0, 1.0, 1.0], [0.0, 1.0, 0.0, 0.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 0.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 0.0, 1.0, 0.0]]),
            "targets": np.array([1, 1, 1, 1, 1, 1]),
            "token_budget": 64,
            "demo_tokens": 54,
            "recency_bonus": 0.7,
        },
    ]


def build_prompting_ladder():
    return [
        {
            "name": "D1 one prompt",
            "base_logits": np.array([[1.0, 0.0]]),
            "demo_boost": 0.8,
            "targets": np.array([0]),
            "prompt_tokens": 1200,
            "window": 2048,
            "format_penalty": -1.0,
            "cot_success": 0.6,
            "samples": 3,
            "imbalance": 0.0,
        },
        {
            "name": "D2 few-shot set",
            "base_logits": np.array([[0.9, 0.0], [0.2, 0.7], [0.8, 0.1]]),
            "demo_boost": 0.7,
            "targets": np.array([0, 1, 0]),
            "prompt_tokens": 900,
            "window": 2048,
            "format_penalty": -0.5,
            "cot_success": 0.62,
            "samples": 3,
            "imbalance": 0.1,
        },
        {
            "name": "D3 distractors and label imbalance",
            "base_logits": np.array([[0.4, 0.2], [0.1, 0.3], [0.5, 0.3], [0.2, 0.4]]),
            "demo_boost": 0.5,
            "targets": np.array([0, 1, 0, 1]),
            "prompt_tokens": 1500,
            "window": 2048,
            "format_penalty": -1.0,
            "cot_success": 0.55,
            "samples": 5,
            "imbalance": 0.6,
        },
        {
            "name": "D4 real-style QA/instruction corpus",
            "base_logits": np.array([[0.9, 0.1], [0.2, 0.8], [0.7, 0.2], [0.1, 0.7], [0.8, 0.0]]),
            "demo_boost": 0.6,
            "targets": np.array([0, 1, 0, 1, 0]),
            "prompt_tokens": 1300,
            "window": 4096,
            "format_penalty": -0.4,
            "cot_success": 0.66,
            "samples": 5,
            "imbalance": 0.0,
        },
        {
            "name": "D5 longer context with ungrounded chains",
            "base_logits": np.array([[0.5, 0.4], [0.4, 0.5], [0.6, 0.5], [0.3, 0.4], [0.5, 0.5], [0.4, 0.6]]),
            "demo_boost": 0.3,
            "targets": np.array([0, 1, 0, 1, 0, 1]),
            "prompt_tokens": 3800,
            "window": 4096,
            "format_penalty": -1.2,
            "cot_success": 0.52,
            "samples": 7,
            "imbalance": 0.8,
        },
    ]


def build_reasoning_ladder():
    return [
        {
            "name": "D1 one prompt",
            "votes": ["A", "A", "B", "A", "B"],
            "correct": "A",
            "branching": 3,
            "depth": 2,
            "checks": [("path1", "claim_shared"), ("path1", "claim_a"), ("path2", "claim_shared"), ("path2", "claim_b")],
            "prior": 0.55,
            "likelihood_ratio": 3.0,
            "tokens_per_trace": 100,
        },
        {
            "name": "D2 few-shot reasoning set",
            "votes": ["A", "C", "A", "A", "B", "A"],
            "correct": "A",
            "branching": 2,
            "depth": 3,
            "checks": [("p1", "sum"), ("p2", "sum"), ("p3", "unit"), ("p4", "lookup")],
            "prior": 0.60,
            "likelihood_ratio": 2.0,
            "tokens_per_trace": 80,
        },
        {
            "name": "D3 distractor correlated errors",
            "votes": ["B", "B", "A", "B", "A", "B"],
            "correct": "A",
            "branching": 3,
            "depth": 2,
            "checks": [("p1", "bad_hint"), ("p2", "bad_hint"), ("p3", "arithmetic"), ("p4", "bad_hint"), ("p5", "arithmetic")],
            "prior": 0.45,
            "likelihood_ratio": 1.5,
            "tokens_per_trace": 90,
        },
        {
            "name": "D4 real-style arithmetic/tool set",
            "votes": ["A", "A", "A", "C", "A", "B", "A"],
            "correct": "A",
            "branching": 3,
            "depth": 3,
            "checks": [("p1", "calc"), ("p2", "calc"), ("p3", "unit"), ("p4", "calendar"), ("p5", "unit"), ("p6", "lookup")],
            "prior": 0.62,
            "likelihood_ratio": 2.8,
            "tokens_per_trace": 110,
        },
        {
            "name": "D5 longer context with weak scorer",
            "votes": ["B", "B", "B", "A", "A", "B", "C", "B"],
            "correct": "A",
            "branching": 4,
            "depth": 3,
            "checks": [("p1", "misread"), ("p2", "misread"), ("p3", "misread"), ("p4", "calc"), ("p5", "lookup"), ("p6", "misread"), ("p7", "calc"), ("p8", "lookup")],
            "prior": 0.50,
            "likelihood_ratio": 1.2,
            "tokens_per_trace": 140,
        },
    ]


def ladder_frame(ladder):
    rows = []
    for rung in ladder:
        keys = [key for key in rung.keys() if key != "name"]
        size = 0
        for value in rung.values():
            if isinstance(value, np.ndarray):
                size = max(size, int(value.size))
        rows.append({"rung": rung["name"], "size": size, "fields": ", ".join(keys[:5])})
    return pd.DataFrame(rows)


We build a prompt-conditioned predictor. It starts with answer logits, adds a demonstration boost to the target option, applies optional format penalties, and computes majority success for independent reasoning samples.

In [ ]:

def majority_success_probability(success_rate, samples):
    needed = samples // 2 + 1
    total = 0.0
    for wins in range(needed, samples + 1):
        ways = math.comb(samples, wins)
        total = total + ways * success_rate ** wins * (1.0 - success_rate) ** (samples - wins)
    return total


def prompt_conditioned_predict(logits, target_index, demo_boost=0.0, format_penalty=0.0):
    adjusted = np.asarray(logits, dtype=float).copy()
    adjusted[target_index] = adjusted[target_index] + demo_boost + format_penalty
    probs = softmax(adjusted)
    return {
        "adjusted": adjusted,
        "probs": probs,
        "prediction": int(np.argmax(probs)),
    }


The lesson formula is $p(y\mid \mathrm{prompt},x)=\mathrm{LM}(y;[\mathrm{instructions},\mathrm{examples},x])$. Chain-of-thought sampling aggregates multiple conditioned draws rather than trusting one trace.

Zero-shot logits $[1,0]$ give probability $0.731$. A demo boost $+0.8$ raises it to $\sigma(1.8)=0.858$. Three independent chain samples with success rate $0.6$ have majority success $0.648$. A 1200-token prompt in a 2048 window leaves 848 tokens, and a bad format penalty $-1.0$ changes odds by $e^{-1}=0.368$.

In [ ]:

zero = prompt_conditioned_predict(np.array([1.0, 0.0]), 0)
few = prompt_conditioned_predict(np.array([1.0, 0.0]), 0, demo_boost=0.8)
majority = majority_success_probability(0.6, 3)
remaining = 2048 - 1200
odds_change = math.exp(-1.0)
assert round(float(zero["probs"][0]), 3) == 0.731
assert round(float(few["probs"][0]), 3) == 0.858
assert round(majority, 3) == 0.648
assert remaining == 848
assert round(odds_change, 3) == 0.368
print(few)


## The dataset ladder
Build the F8 D1-D5 ladder inline. Each rung increases context, distractors, policy pressure, or reasoning complexity while staying tiny and CPU-only.

In [ ]:

ladder = make_f8_ladder("prompting")
preview = ladder_frame(ladder)
print(preview.to_string(index=False))
print("sample rung")
print(ladder[0])


## Run the same method across D1-D5
The same method is applied to every rung; only the rung data changes.

In [ ]:

def evaluate_prompting_rung(rung, balanced=False):
    predictions = []
    costs = []
    for logits, target in zip(rung["base_logits"], rung["targets"]):
        imbalance_penalty = 0.0 if balanced else -rung["imbalance"]
        pred = prompt_conditioned_predict(logits, int(target), demo_boost=rung["demo_boost"], format_penalty=imbalance_penalty)
        predictions.append(pred["prediction"])
        costs.append(rung["prompt_tokens"] + rung["samples"] * 80)
    accuracy = float(np.mean(np.asarray(predictions) == rung["targets"]))
    majority = majority_success_probability(rung["cot_success"], rung["samples"])
    remaining = rung["window"] - rung["prompt_tokens"]
    return {
        "accuracy": accuracy,
        "majority_accuracy": float(majority),
        "tokens": float(np.mean(costs)),
        "remaining_tokens": int(remaining),
    }


results = []
for rung in ladder:
    row = evaluate_prompting_rung(rung)
    row["rung"] = rung["name"]
    results.append(row)

results_df = pd.DataFrame(results)
print(results_df[["rung", "accuracy", "majority_accuracy", "tokens", "remaining_tokens"]].to_string(index=False))


## Results visualization
First inspect per-rung artifacts, then a summary curve for the plan metric.

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for index, rung in enumerate(ladder):
    components = [rung["base_logits"].mean(), rung["demo_boost"], -rung["imbalance"], rung["format_penalty"]]
    axes[index].bar(["base", "demo", "imb", "format"], components)
    axes[index].tick_params(axis="x", rotation=45)
    axes[index].set_title(rung["name"].split()[0])
fig.suptitle("Per-rung prompt component effects")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
context_tokens = [rung["prompt_tokens"] for rung in ladder]
ax.plot(context_tokens, results_df["accuracy"], marker="o", label="direct")
ax.plot(context_tokens, results_df["majority_accuracy"], marker="s", label="CoT majority")
ax.set_xlabel("prompt tokens")
ax.set_ylabel("accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Accuracy versus context tokens")
ax.legend()
plt.show()


## Pitfall on the hardest rung
Pitfall on D5: few-shot label imbalance biases the conditional distribution. Reproduce the skew, then balance or shuffle examples so the prompt does not lean toward one label.

In [ ]:

d5 = ladder[-1]
skewed = evaluate_prompting_rung(d5, balanced=False)
balanced = evaluate_prompting_rung(d5, balanced=True)
print("skewed accuracy", round(skewed["accuracy"], 3))
print("balanced accuracy", round(balanced["accuracy"], 3))
print("skewed majority accuracy", round(skewed["majority_accuracy"], 3))
print("remaining tokens", skewed["remaining_tokens"])



## Evaluate it + Practice
- Metric: answer accuracy per prompt cost; compare against a no-skill baseline such as always choosing the reference/default answer.
- Sanity check: D1 must reproduce the exact lesson arithmetic asserted above before you trust D5.
- Ablation: turn off the key idea (KL anchor, reference margin, helpfulness term, retrieved examples, balanced prompt, or diversified traces) and confirm the metric drops or the failure mode appears.
- Failure signals: saturated probabilities, zero token budget, high reward with low human win rate, or improved safety with collapsed helpfulness.
- Keep everything CPU-only and seeded; do not download models or execute training-heavy notebook code.


Practice: Change the number of chain samples and plot majority accuracy.

Practice: Increase prompt length until no reasoning budget remains.

Practice: Compare zero-shot, one-shot, and balanced few-shot logits on D3.